In [ ]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import glob

from data_processing.score_feature_dataset import (
    ScoreFeatureDataset,
    create_score_feature_dataset_bcss,
)
from flows.shift_flow import ScoreShiftFlowWrapper
from utils.visualize_distributions import plot_score_distribution_with_decoys

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE         = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES    = 5
FEATURE_DIM    = 64
PATH_DATA      = '../../data/BCSS/training/bcss.mini.training.torch'
TEST_FOLDER    = '../../data/BCSS/test/'
FLOWS_PATH     = 'BCSS/bcss_score_shift_flow.pth'
SUBSAMPLE_STEP = 10
MIN_PIXELS     = 50

os.makedirs('BCSS/figures/accuracy',    exist_ok=True)
os.makedirs('BCSS/figures/diagnostics', exist_ok=True)
os.makedirs('BCSS/figures/comparison',  exist_ok=True)

plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'legend.fontsize': 10, 'figure.titlesize': 18,
})
print(f"Device: {DEVICE}")


# ============================================================================
# ДАННЫЕ
# ============================================================================

def get_scores_from_ds(dataset):
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    cnn_list, lbl_list = [], []
    with torch.no_grad():
        for cnn_scores, features, target_decoy, labels in loader:
            cnn_list.append(cnn_scores.cpu().numpy())
            lbl_list.append(labels.cpu().numpy())
    return np.concatenate(cnn_list), np.concatenate(lbl_list)


def load_bcss_test_file(fpath):
    data        = torch.load(fpath, weights_only=False)
    predictions = torch.flatten(data['predictions'], start_dim=2).squeeze(0).T
    features    = torch.flatten(data['features'],    start_dim=2).squeeze(0).T
    labels      = torch.flatten(torch.tensor(data['mask']), start_dim=0)
    labels      = torch.where(labels <= 3, labels, torch.tensor(4))
    return ScoreFeatureDataset(predictions, features, predictions, labels)


# ============================================================================
# BASELINES
# ============================================================================

def _to_tensor(x):
    return torch.from_numpy(x).float() if isinstance(x, np.ndarray) else x


def negentropy(logits):
    if isinstance(logits, np.ndarray):
        logits = torch.from_numpy(logits).float()
    probs   = torch.softmax(logits, dim=1)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=1)
    return np.log(logits.shape[1]) - entropy


def predict_ATC_maxconf(source_logits, source_labels, target_logits):
    source_logits = _to_tensor(source_logits)
    source_labels = _to_tensor(source_labels).long()
    target_logits = _to_tensor(target_logits)
    src_scores    = torch.softmax(source_logits, dim=1).amax(1)
    tgt_scores    = torch.softmax(target_logits, dim=1).amax(1)
    n_correct     = (source_logits.argmax(1) == source_labels).sum()
    threshold     = torch.sort(src_scores)[0][-n_correct]
    return (tgt_scores > threshold).float().mean().item()


def predict_ATC_negent(source_logits, source_labels, target_logits):
    source_logits = _to_tensor(source_logits)
    source_labels = _to_tensor(source_labels).long()
    target_logits = _to_tensor(target_logits)
    src_scores    = negentropy(source_logits)
    tgt_scores    = negentropy(target_logits)
    n_correct     = (source_logits.argmax(1) == source_labels).sum()
    threshold     = torch.sort(src_scores)[0][-n_correct]
    return (tgt_scores > threshold).float().mean().item()


def predict_AC(source_logits, source_labels, target_logits):
    return _to_tensor(target_logits).softmax(dim=1).amax(1).mean().item()


def predict_DOC(source_logits, source_labels, target_logits):
    source_logits = _to_tensor(source_logits)
    source_labels = _to_tensor(source_labels).long()
    target_logits = _to_tensor(target_logits)
    src_conf = torch.softmax(source_logits, dim=1).amax(1).mean().item()
    tgt_conf = torch.softmax(target_logits, dim=1).amax(1).mean().item()
    src_acc  = (source_logits.argmax(1) == source_labels).float().mean().item()
    return src_acc + (tgt_conf - src_conf)


try:
    import ot
    def predict_COT(source_logits, source_labels, target_logits):
        source_logits  = _to_tensor(source_logits)
        source_labels  = _to_tensor(source_labels).long()
        target_logits  = _to_tensor(target_logits)
        num_classes    = source_logits.shape[1]
        src_label_dist = torch.nn.functional.one_hot(
            source_labels, num_classes).float().mean(0)
        target_probs   = torch.softmax(target_logits, dim=1)
        cost_matrix    = torch.stack([
            (target_probs - onehot).abs().sum(1)
            for onehot in torch.eye(num_classes)
        ], dim=1) / 2
        ot_plan = ot.emd(
            np.ones(len(target_probs)) / len(target_probs),
            src_label_dist.cpu().numpy(),
            cost_matrix.cpu().numpy()
        )
        ot_cost  = np.sum(ot_plan * cost_matrix.cpu().numpy())
        s_conf   = torch.softmax(source_logits, dim=1).amax(1).mean().item()
        s_acc    = (source_logits.argmax(1) == source_labels).float().mean().item()
        return 1.0 - (ot_cost + s_conf - s_acc)
    COT_AVAILABLE = True
    print("✓ COT available")
except ImportError:
    COT_AVAILABLE = False
    print("⚠ COT unavailable")

BASELINE_METHODS = {
    'ATC'   : predict_ATC_maxconf,
    'ATC-NE': predict_ATC_negent,
    'AC'    : predict_AC,
    'DOC'   : predict_DOC,
}
if COT_AVAILABLE:
    BASELINE_METHODS['COT'] = predict_COT

print(f"Baselines: {list(BASELINE_METHODS.keys())}")


# ============================================================================
# MIX-MAX FDR
#
#              SUM_{Z_j > s_th} [ P(f(t) <= Z_j) / P(g(t) <= Z_j) ]_{[0,1]}
# FDP_MM  =  ────────────────────────────────────────────────────────────────
#                              1{ f(t) >= s_th }
#
# f(t) = score предсказанного класса
# Z_j  = decoy score того же класса
# g(t) = max(f(t), Z_j)
# ============================================================================

def calculate_mixmax_qvalues(model_scores: np.ndarray,
                              decoy_scores: np.ndarray,
                              verbose: bool = True) -> np.ndarray:
    n            = len(model_scores)
    pred_classes = model_scores.argmax(axis=1)
    f_t          = model_scores[np.arange(n), pred_classes]
    Z            = decoy_scores[np.arange(n), pred_classes]
    g_t          = np.maximum(f_t, Z)

    if verbose:
        overlap = ((Z >= f_t.min()) & (Z <= f_t.max())).mean()
        print(f"  f(t) : [{f_t.min():.3f}, {f_t.max():.3f}]  "
              f"mean={f_t.mean():.3f}")
        print(f"  Z    : [{Z.min():.3f},  {Z.max():.3f}]  "
              f"mean={Z.mean():.3f}")
        print(f"  Z in f(t) range: {overlap:.3f}  "
              f"target wins: {(f_t > Z).mean():.3f}")

    unique_Z, counts_Z = np.unique(Z, return_counts=True)
    sorted_f = np.sort(f_t)
    sorted_g = np.sort(g_t)

    P_F = np.searchsorted(sorted_f, unique_Z, side='right') / n
    P_G = np.searchsorted(sorted_g, unique_Z, side='right') / n

    with np.errstate(divide='ignore', invalid='ignore'):
        R_j = np.where(
            (P_G > 0) & (P_F > 0),
            np.clip(P_F / P_G, 0.0, 1.0),
            0.0
        )

    n_unique      = len(unique_Z)
    order         = np.argsort(f_t)[::-1]
    sorted_f_desc = f_t[order]
    fdr_values    = np.zeros(n)

    for i, s_th in enumerate(sorted_f_desc):
        D         = i + 1
        idx_start = np.searchsorted(unique_Z, s_th, side='right')
        numerator = float(np.sum(R_j[idx_start:] * counts_Z[idx_start:])) \
                    if idx_start < n_unique else 0.0
        fdr_values[i] = numerator / D

    if verbose:
        print(f"  FDP range: [{fdr_values.min():.4f}, {fdr_values.max():.4f}]")

    q_desc         = np.minimum.accumulate(fdr_values[::-1])[::-1]
    final_q        = np.empty(n)
    final_q[order] = q_desc
    return final_q


def _compute_gt_qvalues(pred_classes, f_t, labels):
    n     = len(f_t)
    order = np.argsort(f_t)[::-1]
    n_incorrect, fdr_gt = 0, np.zeros(n)
    for rank, idx in enumerate(order):
        if pred_classes[idx] != labels[idx]:
            n_incorrect += 1
        fdr_gt[rank] = n_incorrect / (rank + 1)
    for i in range(n - 2, -1, -1):
        fdr_gt[i] = min(fdr_gt[i], fdr_gt[i + 1])
    final_q        = np.empty(n)
    final_q[order] = fdr_gt
    return final_q


def control_fdr_mixmax(model_scores, target_labels, decoy_scores,
                        verbose=True):
    n            = len(model_scores)
    pred_classes = model_scores.argmax(axis=1)
    f_t          = model_scores[np.arange(n), pred_classes]
    df = pd.DataFrame({
        'original_index'  : np.arange(n),
        'label'           : target_labels,
        'predicted_class' : pred_classes,
        'pred_class_score': f_t,
    })
    df['q_values_mixmax']       = calculate_mixmax_qvalues(
        model_scores, decoy_scores, verbose=verbose)
    df['q_values_ground_truth'] = _compute_gt_qvalues(
        pred_classes, f_t, target_labels)
    return df


# ============================================================================
# ESTIMATION CURVE
#
# π₀ = FDP_MM(-∞) = q-value последнего сэмпла (все приняты)
#
# FP(s_th) = D * FDP(s_th)
# TP(s_th) = D * (1 - FDP(s_th))
# TN(s_th) = |T| * π₀ - FP(s_th)
# FN(s_th) = |T| * (1 - π₀) - TP(s_th)
# ACC(s_th) = (TP + TN) / |T|
# ACC_ST   = ACC(s_th = -∞)
# ACC_TA   = max_{s_th} ACC(s_th)
# ============================================================================

def compute_method_estimation_curve(df: pd.DataFrame,
                                     q_col: str) -> pd.DataFrame:
    total    = len(df)
    true_pi0 = 1.0 - float((df['predicted_class'] == df['label']).mean())
    true_acc_full = 1.0 - true_pi0

    df_m = df[~df[q_col].isna()].copy()
    if len(df_m) == 0:
        return pd.DataFrame()

    # ── Сортировка по q возрастая (определяет estimated кривую) ──────────
    df_m = df_m.sort_values(q_col, ascending=True).reset_index(drop=True)
    df_m['n_discoveries'] = np.arange(1, len(df_m) + 1)

    # ── π₀_est = FDP_MM(-∞) ──────────────────────────────────────────────
    pi0_est = float(np.clip(df_m[q_col].iloc[-1], 0.0, 1.0))
    print(f"  π₀_est  = FDP_MM(-∞) = {pi0_est:.4f}")
    print(f"  π₀_true = true error = {true_pi0:.4f}")

    # ── Label-free estimated confusion matrix ─────────────────────────────
    df_m['FP_est'] = df_m['n_discoveries'] * df_m[q_col]
    df_m['TP_est'] = df_m['n_discoveries'] - df_m['FP_est']
    df_m['TN_est'] = total * pi0_est - df_m['FP_est']
    df_m['FN_est'] = total * (1.0 - pi0_est) - df_m['TP_est']
    for col in ('FP_est', 'TP_est', 'TN_est', 'FN_est'):
        df_m[col] = np.maximum(0.0, df_m[col])
    df_m['Accuracy_est'] = (df_m['TP_est'] + df_m['TN_est']) / total

    # ── True confusion matrix — строим по убыванию f(t), НЕ по q ─────────
    # f(t) одинаковый для любых decoys → синяя кривая одинаковая всегда
    df_true = df[['predicted_class', 'label', 'pred_class_score']].copy()
    df_true = df_true.sort_values(
        'pred_class_score', ascending=False).reset_index(drop=True)

    df_true['is_correct'] = (
        df_true['predicted_class'] == df_true['label']).astype(int)
    n_disc_true        = np.arange(1, len(df_true) + 1)
    df_true['TP_true'] = df_true['is_correct'].cumsum().astype(float)
    df_true['FP_true'] = n_disc_true - df_true['TP_true']
    df_true['TN_true'] = np.maximum(
        0.0, total * true_pi0 - df_true['FP_true'])
    df_true['Accuracy_true_at_threshold'] = (
        df_true['TP_true'] + df_true['TN_true']) / total

    # Маппинг: pred_class_score → true accuracy
    # (у каждого уникального f(t) своя строка в df_true)
    score_to_true_acc = dict(zip(
        df_true['pred_class_score'].values,
        df_true['Accuracy_true_at_threshold'].values,
    ))

    # Добавляем в df_m через pred_class_score
    df_m['Accuracy_true_at_threshold'] = df_m['pred_class_score'].map(
        score_to_true_acc)

    df_m['true_acc_full']   = true_acc_full
    df_m['pi0_est']         = pi0_est
    df_m['true_pi0']        = true_pi0

    df_m['error_est_vs_thresh'] = np.abs(
        df_m['Accuracy_est'] - df_m['Accuracy_true_at_threshold'])
    df_m['error_est_vs_full'] = np.abs(
        df_m['Accuracy_est'] - true_acc_full)

    return df_m.rename(columns={q_col: 'q_value_method'})


def find_best_mixmax_threshold(df_curve: pd.DataFrame) -> dict:
    if len(df_curve) == 0:
        return dict(pi0_est=np.nan,
                    acc_st_est=np.nan, acc_ta_est=np.nan,
                    acc_st_true=np.nan, acc_ta_true=np.nan,
                    err_st=np.nan, err_ta=np.nan,
                    best_q=np.nan, n_accepted=0)

    pi0      = float(df_curve['pi0_est'].iloc[0])

    # ACC_ST = ACC(-∞): последняя строка (все приняты)
    last     = df_curve.iloc[-1]
    acc_st_est  = float(last['Accuracy_est'])
    acc_st_true = float(last['true_acc_full'])

    # ACC_TA = max ACC_est
    best_idx    = df_curve['Accuracy_est'].idxmax()
    best        = df_curve.loc[best_idx]
    acc_ta_est  = float(best['Accuracy_est'])
    acc_ta_true = float(best['Accuracy_true_at_threshold'])

    return dict(
        pi0_est     = pi0,
        acc_st_est  = acc_st_est,
        acc_ta_est  = acc_ta_est,
        acc_st_true = acc_st_true,
        acc_ta_true = acc_ta_true,
        err_st      = abs(acc_st_est  - acc_st_true),
        err_ta      = abs(acc_ta_est  - acc_ta_true),
        best_q      = float(best['q_value_method']),
        n_accepted  = int(best['n_discoveries']),
    )


# ============================================================================
# EMPIRICAL DECOYS
# ============================================================================

def build_negative_pools(train_scores, train_labels, num_classes):
    """scores[:, c] для сэмплов где label != c"""
    pools = {}
    for c in range(num_classes):
        mask     = train_labels != c
        pools[c] = train_scores[mask, c]
        print(f"  Class {c}: {len(pools[c])} neg scores  "
              f"[{pools[c].min():.2f}, {pools[c].max():.2f}]")
    return pools


def generate_empirical_decoys(model_scores, neg_pools, seed=42):
    """decoy[:, c] ~ P(score_c | label != c)"""
    rng = np.random.default_rng(seed)
    n, C = model_scores.shape
    decoys = np.zeros_like(model_scores)
    for c in range(C):
        decoys[:, c] = rng.choice(neg_pools[c], size=n, replace=True)
    return decoys


# ============================================================================
# ДИАГНОСТИКА
# ============================================================================

def diagnose_tile(model_scores, decoy_scores, labels, tile_name="tile"):
    n            = len(model_scores)
    pred_classes = model_scores.argmax(axis=1)
    f_t          = model_scores[np.arange(n), pred_classes]
    Z            = decoy_scores[np.arange(n), pred_classes]
    g_t          = np.maximum(f_t, Z)
    is_correct   = (pred_classes == labels)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
    fig.suptitle(f'Diagnostics: {tile_name}', fontweight='bold')

    # 1. Target vs Decoy distribution
    ax   = axes[0, 0]
    bins = np.linspace(min(f_t.min(), Z.min()), max(f_t.max(), Z.max()), 80)
    ax.hist(f_t, bins=bins, alpha=0.6, color='#E53935', density=True,
            label=f'Target f(t)  mean={f_t.mean():.2f}')
    ax.hist(Z,   bins=bins, alpha=0.6, color='#1976D2', density=True,
            label=f'Decoy Z  mean={Z.mean():.2f}')
    ax.set_xlabel('Score'); ax.set_ylabel('Density')
    ax.set_title('Target vs Decoy'); ax.legend(fontsize=9)

    # 2. Win rate per class
    ax = axes[0, 1]
    win_rates, class_sizes = [], []
    for c in range(model_scores.shape[1]):
        mask = pred_classes == c
        win_rates.append((f_t[mask] > Z[mask]).mean() if mask.sum() > 0 else 0)
        class_sizes.append(mask.sum())
    bars = ax.bar(np.arange(model_scores.shape[1]), win_rates,
                  color='#388E3C', alpha=0.7, edgecolor='black')
    ax.axhline(0.5, color='red', ls='--', lw=1.5, label='Random (0.5)')
    ax.axhline(is_correct.mean(), color='black', ls='--', lw=1.5,
               label=f'True ACC={is_correct.mean():.3f}')
    for bar, sz in zip(bars, class_sizes):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01, f'n={sz}', ha='center', fontsize=8)
    ax.set_xlabel('Predicted Class'); ax.set_ylabel('Win Rate')
    ax.set_title('Target Win Rate per Class')
    ax.set_ylim(0, 1.1); ax.legend(fontsize=9)

    # 3. CDF
    ax = axes[0, 2]
    p  = np.linspace(0, 1, n)
    ax.plot(np.sort(f_t), p, color='#E53935', lw=2, label='CDF f(t)')
    ax.plot(np.sort(Z),   p, color='#1976D2', lw=2, label='CDF Z')
    ax.plot(np.sort(g_t), p, color='#7B1FA2', lw=2, ls='--', label='CDF g(t)')
    ax.set_xlabel('Score'); ax.set_ylabel('CDF')
    ax.set_title('CDF: Target / Decoy / Mix-Max')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # 4. Scatter f(t) vs Z
    ax    = axes[1, 0]
    sub   = 2000
    idx_c = np.where( is_correct)[0]
    idx_w = np.where(~is_correct)[0]
    idx_c = idx_c[np.random.choice(len(idx_c), min(sub, len(idx_c)), replace=False)]
    idx_w = idx_w[np.random.choice(len(idx_w), min(sub, len(idx_w)), replace=False)]
    ax.scatter(f_t[idx_c], Z[idx_c], alpha=0.3, s=5, color='#388E3C', label='Correct')
    ax.scatter(f_t[idx_w], Z[idx_w], alpha=0.3, s=5, color='#E53935', label='Wrong')
    lo, hi = min(f_t.min(), Z.min()), max(f_t.max(), Z.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, alpha=0.5, label='f(t)=Z')
    ax.set_xlabel('Target f(t)'); ax.set_ylabel('Decoy Z')
    ax.set_title('Scatter: Target vs Decoy')
    ax.legend(fontsize=8, markerscale=3)

    # 5. R_j
    ax = axes[1, 1]
    unique_Z_, _ = np.unique(Z, return_counts=True)
    P_F_ = np.searchsorted(np.sort(f_t), unique_Z_, side='right') / n
    P_G_ = np.searchsorted(np.sort(g_t), unique_Z_, side='right') / n
    R_j_ = np.where((P_G_ > 0) & (P_F_ > 0),
                    np.clip(P_F_ / P_G_, 0, 1), 0)
    ax.plot(unique_Z_, R_j_, color='#E53935', lw=1.5)
    ax.axhline(1.0, color='gray', ls='--', lw=1)
    ax.axhline(0.5, color='gray', ls=':', lw=1)
    ax.set_xlabel('Decoy Z'); ax.set_ylabel('R_j = P(f≤Z)/P(g≤Z)')
    ax.set_title('Mix-Max R_j weights'); ax.grid(True, alpha=0.3)

    # 6. FDP curve
    ax    = axes[1, 2]
    q_mm  = calculate_mixmax_qvalues(model_scores, decoy_scores, verbose=False)
    order = np.argsort(f_t)[::-1]
    n_disc = np.arange(1, n + 1)
    ax.plot(n_disc / n, q_mm[order], color='#E53935', lw=2,
            label='Mix-Max FDP est')
    is_wrong_sorted = (~is_correct[order]).astype(int)
    fdp_gt = np.cumsum(is_wrong_sorted) / n_disc
    for i in range(len(fdp_gt) - 2, -1, -1):
        fdp_gt[i] = min(fdp_gt[i], fdp_gt[i + 1])
    ax.plot(n_disc / n, fdp_gt, color='black', lw=2, ls='--', label='True FDP')
    ax.set_xlabel('Fraction accepted'); ax.set_ylabel('FDP')
    ax.set_title('FDP: Mix-Max vs True')
    ax.set_ylim(0, 1); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    print(f"\n{'='*50}\nDIAGNOSTICS: {tile_name}\n{'='*50}")
    print(f"  n_pixels        : {n}")
    print(f"  true_accuracy   : {is_correct.mean():.4f}")
    print(f"  target_win_rate : {(f_t > Z).mean():.4f}")
    print(f"  f(t) mean/std   : {f_t.mean():.3f} / {f_t.std():.3f}")
    print(f"  Z   mean/std    : {Z.mean():.3f} / {Z.std():.3f}")
    print(f"  R_j mean        : {R_j_.mean():.4f}")

    for ext in ('png', 'pdf'):
        plt.savefig(f'BCSS/figures/diagnostics/diagnose_{tile_name}.{ext}',
                    dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved")


# ============================================================================
# ГРАФИК ACCURACY
# ============================================================================

def plot_accuracy_curve(df_curve, metrics, baseline_results,
                         title='', save_path=None):
    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)

    q   = df_curve['q_value_method'].values
    est = df_curve['Accuracy_est'].values
    tru = df_curve['Accuracy_true_at_threshold'].values

    ax.plot(q, est, color='#E53935', lw=2.5, label='Mix-Max est (label-free)')
    ax.plot(q, tru, color='#1976D2', lw=2.0, ls='-.', label='True ACC @ threshold')
    ax.axhline(metrics['acc_st_true'], color='black', lw=2, ls='--',
               label=f"True ACC_ST = {metrics['acc_st_true']:.3f}")

    best_q = metrics['best_q']
    ax.axvline(best_q, color='gray', ls=':', lw=1.5)
    ax.scatter([best_q], [metrics['acc_ta_est']],  color='#E53935', s=90, zorder=5,
               label=f"ACC_TA est={metrics['acc_ta_est']:.3f}")
    ax.scatter([best_q], [metrics['acc_ta_true']], color='#1976D2', s=90, zorder=5,
               label=f"ACC_TA true={metrics['acc_ta_true']:.3f}")

    bl_colors = ['#388E3C', '#0288D1', '#7B1FA2', '#F57C00', '#00838F']
    for (name, val), col in zip(baseline_results.items(), bl_colors):
        if not np.isnan(val):
            ax.axhline(val, color=col, lw=1.5, ls='--', alpha=0.7,
                       label=f'{name}={val:.3f}')

    for fdr in [0.05, 0.10, 0.20]:
        ax.axvline(fdr, color='gray', ls=':', alpha=0.25, lw=1)
        ax.text(fdr + 0.002, 0.02, f'{fdr:.2f}', fontsize=8, alpha=0.5)

    ax.set_xlabel('Q-value (FDR threshold)', fontweight='bold')
    ax.set_ylabel('Accuracy', fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0, min(0.55, float(q.max()) + 0.03))
    ax.legend(fontsize=8, framealpha=0.9, loc='lower left')
    ax.grid(True, alpha=0.3, ls='--')
    ax.set_title(title, fontweight='bold')

    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        for ext in ('png', 'pdf'):
            plt.savefig(f'{save_path}.{ext}', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {save_path}")


def print_metrics(met, title=""):
    print(f"\n  [{title}]")
    print(f"  π₀  = {met['pi0_est']:.4f}")
    print(f"  ACC_ST: est={met['acc_st_est']:.4f}  "
          f"true={met['acc_st_true']:.4f}  err={met['err_st']:.4f}")
    print(f"  ACC_TA: est={met['acc_ta_est']:.4f}  "
          f"true={met['acc_ta_true']:.4f}  err={met['err_ta']:.4f}")
    print(f"  best_q={met['best_q']:.4f}  "
          f"n_accepted={met['n_accepted']}")


# ============================================================================
# ЗАГРУЗКА ДАННЫХ И FLOW
# ============================================================================

print("\n" + "="*70)
print("LOADING TRAINING DATA")
print("="*70)
data     = torch.load(PATH_DATA)
train_ds = create_score_feature_dataset_bcss(data, DEVICE)
train_scores, train_labels = get_scores_from_ds(train_ds)
print(f"Train: {train_scores.shape}")
for c in range(NUM_CLASSES):
    print(f"  Class {c}: {(train_labels == c).sum()}")

print("\n" + "="*70)
print("FLOW MODEL")
print("="*70)
flow = ScoreShiftFlowWrapper(
    num_classes=NUM_CLASSES, n_flows=12,
    feature_dim=FEATURE_DIM, hidden_dim=256, encoder_dim=128,
).to(DEVICE)
flow.load_state_dict(torch.load(FLOWS_PATH, map_location=DEVICE))
flow.eval()
print("✓ Flow loaded")

test_files = sorted(glob.glob(os.path.join(TEST_FOLDER, '*.tensor')))
print(f"Test files: {len(test_files)}")

# ── Negative pools для empirical decoys ──────────────────────────────────────
print("\nBuilding negative pools from train...")
neg_pools_train = build_negative_pools(train_scores, train_labels, NUM_CLASSES)

# ============================================================================
# TILE 0
# ============================================================================

print("\n" + "="*70)
print("TILE 0")
print("="*70)

test_ds_0      = load_bcss_test_file(test_files[0])
ms0, ds_flow0, ls0 = flow.generate_decoys(test_ds_0, device=DEVICE)
ms0       = ms0      [::SUBSAMPLE_STEP]
ds_flow0  = ds_flow0 [::SUBSAMPLE_STEP]
ls0       = ls0      [::SUBSAMPLE_STEP]

# Baselines (один раз)
bl0 = {}
for name, fn in BASELINE_METHODS.items():
    try:    bl0[name] = fn(train_scores, train_labels, ms0)
    except: bl0[name] = np.nan
print("Baselines tile 0:", {k: f"{v:.4f}" for k, v in bl0.items()})

# ── Flow decoys ───────────────────────────────────────────────────────────────
print("\n--- Flow decoys ---")
diagnose_tile(ms0, ds_flow0, ls0, tile_name='tile0_flow')
df_mm   = control_fdr_mixmax(ms0, ls0, ds_flow0)
df_cur  = compute_method_estimation_curve(df_mm, 'q_values_mixmax')
met     = find_best_mixmax_threshold(df_cur)
print_metrics(met, "Tile 0 | Flow")
plot_accuracy_curve(df_cur, met, bl0,
                    title='Tile 0 — Flow Decoys',
                    save_path='BCSS/figures/accuracy/acc_tile0_flow')

# ── Empirical decoys ──────────────────────────────────────────────────────────
print("\n--- Empirical decoys ---")
ds_emp0 = generate_empirical_decoys(ms0, neg_pools_train)
diagnose_tile(ms0, ds_emp0, ls0, tile_name='tile0_empirical')
df_mm_e  = control_fdr_mixmax(ms0, ls0, ds_emp0)
df_cur_e = compute_method_estimation_curve(df_mm_e, 'q_values_mixmax')
met_e    = find_best_mixmax_threshold(df_cur_e)
print_metrics(met_e, "Tile 0 | Empirical")
plot_accuracy_curve(df_cur_e, met_e, bl0,
                    title='Tile 0 — Empirical Decoys',
                    save_path='BCSS/figures/accuracy/acc_tile0_empirical')

# ============================================================================
# TILE 1
# ============================================================================

print("\n" + "="*70)
print("TILE 1")
print("="*70)

test_ds_1      = load_bcss_test_file(test_files[1])
ms1, ds_flow1, ls1 = flow.generate_decoys(test_ds_1, device=DEVICE)
ms1       = ms1      [::SUBSAMPLE_STEP]
ds_flow1  = ds_flow1 [::SUBSAMPLE_STEP]
ls1       = ls1      [::SUBSAMPLE_STEP]

bl1 = {}
for name, fn in BASELINE_METHODS.items():
    try:    bl1[name] = fn(train_scores, train_labels, ms1)
    except: bl1[name] = np.nan
print("Baselines tile 1:", {k: f"{v:.4f}" for k, v in bl1.items()})

# ── Flow decoys ───────────────────────────────────────────────────────────────
print("\n--- Flow decoys ---")
diagnose_tile(ms1, ds_flow1, ls1, tile_name='tile1_flow')
df_mm1   = control_fdr_mixmax(ms1, ls1, ds_flow1)
df_cur1  = compute_method_estimation_curve(df_mm1, 'q_values_mixmax')
met1     = find_best_mixmax_threshold(df_cur1)
print_metrics(met1, "Tile 1 | Flow")
plot_accuracy_curve(df_cur1, met1, bl1,
                    title='Tile 1 — Flow Decoys',
                    save_path='BCSS/figures/accuracy/acc_tile1_flow')

# ── Empirical decoys ──────────────────────────────────────────────────────────
print("\n--- Empirical decoys ---")
ds_emp1  = generate_empirical_decoys(ms1, neg_pools_train)
diagnose_tile(ms1, ds_emp1, ls1, tile_name='tile1_empirical')
df_mm1_e  = control_fdr_mixmax(ms1, ls1, ds_emp1)
df_cur1_e = compute_method_estimation_curve(df_mm1_e, 'q_values_mixmax')
met1_e    = find_best_mixmax_threshold(df_cur1_e)
print_metrics(met1_e, "Tile 1 | Empirical")
plot_accuracy_curve(df_cur1_e, met1_e, bl1,
                    title='Tile 1 — Empirical Decoys',
                    save_path='BCSS/figures/accuracy/acc_tile1_empirical')

# ============================================================================
# ИТОГОВАЯ ТАБЛИЦА
# ============================================================================

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"\n{'Case':<35} {'π₀':>6} {'ST_est':>8} {'ST_true':>8} "
      f"{'err_ST':>7} {'TA_est':>8} {'TA_true':>8} {'err_TA':>7}")
print("-" * 90)

summary = [
    ("Tile 0 | Flow decoys",      met),
    ("Tile 0 | Empirical decoys", met_e),
    ("Tile 1 | Flow decoys",      met1),
    ("Tile 1 | Empirical decoys", met1_e),
]
for name, m in summary:
    print(f"  {name:<33} {m['pi0_est']:>6.3f} "
          f"{m['acc_st_est']:>8.4f} {m['acc_st_true']:>8.4f} "
          f"{m['err_st']:>7.4f} "
          f"{m['acc_ta_est']:>8.4f} {m['acc_ta_true']:>8.4f} "
          f"{m['err_ta']:>7.4f}")

print("\nBaselines vs True ACC_ST:")
print(f"  {'Method':<10} {'Tile0_true':>10} {'Tile0_bl':>10} "
      f"{'Tile1_true':>10} {'Tile1_bl':>10}")
print("-" * 50)
for name in BASELINE_METHODS:
    print(f"  {name:<10} {met['acc_st_true']:>10.4f} "
          f"{bl0.get(name, np.nan):>10.4f} "
          f"{met1['acc_st_true']:>10.4f} "
          f"{bl1.get(name, np.nan):>10.4f}")

print("\n✓ Done")

Device: cuda:1
✓ COT available
Baselines: ['ATC', 'ATC-NE', 'AC', 'DOC', 'COT']

LOADING TRAINING DATA


/tmp/ipykernel_46636/2819914106.py:574: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data     = torch.load(PATH_DATA)


Train: (3394245, 5)
  Class 0: 1000000
  Class 1: 1000000
  Class 2: 776366
  Class 3: 568007
  Class 4: 49872

FLOW MODEL
✓ ScoreShiftFlow defined — single flow over full score vector
✓ Flow loaded
Test files: 48

Building negative pools from train...
  Class 0: 2394245 neg scores  [-11.14, 36.09]
  Class 1: 2394245 neg scores  [-2.19, 32.83]
  Class 2: 2617879 neg scores  [-147.66, 16.53]


/tmp/ipykernel_46636/2819914106.py:588: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  flow.load_state_dict(torch.load(FLOWS_PATH, map_location=DEVICE))


  Class 3: 2826238 neg scores  [-170.91, 8.06]
  Class 4: 3344373 neg scores  [-19.41, 30.54]

TILE 0


In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import glob

from data_processing.score_feature_dataset import (
    ScoreFeatureDataset,
    create_score_feature_dataset_bcss,
)
from data_processing.negative_scores_pool import collect_negative_scores
from flows.shift_flow import ScoreShiftFlowWrapper
from utils.other_methods import *
from utils.visualize_distributions import *
from fdr.fdr_control import *
from fdr.plot_fdr import *

try:
    from utils.other_methods import predict_COT
    COT_AVAILABLE = True
except ImportError:
    COT_AVAILABLE = False

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE         = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES    = 5
FEATURE_DIM    = 64
PATH_DATA      = '../../data/BCSS/training/bcss.mini.training.torch'
TEST_FOLDER    = '../../data/BCSS/test/'
FLOWS_PATH     = 'BCSS/bcss_score_shift_flow_new.pth'
SUBSAMPLE_STEP = 10
MIN_PIXELS     = 50        # минимум пикселей чтобы считать threshold-accuracy

os.makedirs('BCSS/figures/accuracy',    exist_ok=True)
os.makedirs('BCSS/figures/diagnostics', exist_ok=True)
os.makedirs('BCSS/figures/comparison',  exist_ok=True)

plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'legend.fontsize': 10, 'figure.titlesize': 18,
})

BASELINE_METHODS = {
    'ATC'   : predict_ATC_maxconf,
    'ATC-NE': predict_ATC_negent,
    'AC'    : predict_AC,
    'DOC'   : predict_DOC,
}
if COT_AVAILABLE:
    BASELINE_METHODS['COT'] = predict_COT

COLORS = {
    'Mix-Max': '#E53935',
    'ATC'    : '#1976D2',
    'ATC-NE' : '#0288D1',
    'AC'     : '#388E3C',
    'DOC'    : '#7B1FA2',
    'COT'    : '#F57C00',
}

print(f"Device    : {DEVICE}")
print(f"Baselines : {list(BASELINE_METHODS.keys())}")


# ============================================================================
# HELPERS
# ============================================================================

def get_scores_from_ds(dataset):
    """Вытащить scores и labels из ScoreFeatureDataset."""
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    cnn_list, lbl_list = [], []
    with torch.no_grad():
        for cnn_scores, features, target_decoy, labels in loader:
            cnn_list.append(cnn_scores.cpu().numpy())
            lbl_list.append(labels.cpu().numpy())
    return np.concatenate(cnn_list), np.concatenate(lbl_list)


def load_bcss_test_file(fpath):
    """Загрузить один .tensor тайл → ScoreFeatureDataset."""
    data        = torch.load(fpath, weights_only=False)
    predictions = torch.flatten(data['predictions'], start_dim=2).squeeze(0).T
    features    = torch.flatten(data['features'],    start_dim=2).squeeze(0).T
    labels      = torch.flatten(torch.tensor(data['mask']), start_dim=0)
    labels      = torch.where(labels <= 3, labels, torch.tensor(4))
    return ScoreFeatureDataset(predictions, features, predictions, labels)


def true_accuracy_full(model_scores, labels_np):
    """
    Обычная accuracy по всем пикселям.
    Это ground truth с которым сравниваем ВСЕХ.
    """
    pred = model_scores.argmax(axis=1)
    return float((pred == labels_np).mean())


def true_accuracy_at_threshold(model_scores, labels_np, conf_threshold):
    """
    Реальная accuracy только среди пикселей
    где max_score >= conf_threshold.
    Знаменатель = число принятых пикселей.
    """
    pred          = model_scores.argmax(axis=1)
    max_scores    = model_scores[np.arange(len(model_scores)), pred]
    mask          = max_scores >= conf_threshold
    n_accepted    = mask.sum()
    if n_accepted < MIN_PIXELS:
        return np.nan, 0
    acc = float((pred[mask] == labels_np[mask]).mean())
    return acc, int(n_accepted)


def find_best_mixmax_threshold(df_curve):
    """
    Найти q* где Accuracy_est максимальна.
    Вернуть: est_acc, true_acc_at_q, error, q*, n_accepted
    
    df_curve — результат compute_method_estimation_curve().
    Колонки которые нам нужны:
        q_value_method          — q-value порог
        Accuracy_est            — наша оценка accuracy (label-free)
        Accuracy_true_at_threshold — реальная accuracy среди принятых
        error_at_threshold      — |est - true_at_threshold|
        n_discoveries           — сколько пикселей принято
    """
    if len(df_curve) == 0:
        return dict(
            mixmax_est=np.nan,
            mixmax_true_at_q=np.nan,
            mixmax_error=np.nan,
            mixmax_best_q=np.nan,
            mixmax_n_accepted=0,
        )

    best_idx = df_curve['Accuracy_est'].idxmax()
    row      = df_curve.loc[best_idx]

    return dict(
        mixmax_est       = float(row['Accuracy_est']),
        mixmax_true_at_q = float(row['Accuracy_true_at_threshold']),
        mixmax_error     = float(row['error_at_threshold']),
        mixmax_best_q    = float(row['q_value_method']),
        mixmax_n_accepted= int(row['n_discoveries']),
    )


# ============================================================================
# ДАННЫЕ И FLOW
# ============================================================================

print("\n" + "="*70)
print("LOADING TRAINING DATA")
print("="*70)

data     = torch.load(PATH_DATA)
train_ds = create_score_feature_dataset_bcss(data, DEVICE)
train_scores, train_labels = get_scores_from_ds(train_ds)
print(f"Train scores shape: {train_scores.shape}")
for c in range(NUM_CLASSES):
    print(f"  Class {c}: {(train_labels == c).sum()} samples")

print("\n" + "="*70)
print("FLOW MODEL")
print("="*70)

flow = ScoreShiftFlowWrapper(
    num_classes=NUM_CLASSES,
    n_flows=12,
    feature_dim=FEATURE_DIM,
    hidden_dim=256,
    encoder_dim=128,
).to(DEVICE)

if os.path.exists(FLOWS_PATH):
    flow.load_state_dict(torch.load(FLOWS_PATH, map_location=DEVICE))
    print(f"✓ Flow loaded from {FLOWS_PATH}")
else:
    print("Training ScoreShiftFlow...")
    flow.train_flow(
        train_ds, epochs=30, lr=3e-4, batch_size=256,
        device=DEVICE, patience=5, grad_clip=1.0)
    torch.save(flow.state_dict(), FLOWS_PATH)
    print(f"✓ Flow saved to {FLOWS_PATH}")


# ============================================================================
# PLOT 5: Score distributions для первого тайла
# ============================================================================

print("\n" + "="*70)
print("SCORE DISTRIBUTIONS (первый тайл)")
print("="*70)

test_files = sorted(glob.glob(os.path.join(TEST_FOLDER, '*.tensor')))

first_tile = test_files[1]
tile_name  = os.path.basename(first_tile)
print(f"Тайл: {tile_name}")

try:
    test_ds_dist = load_bcss_test_file(first_tile)

    ms, ds_decoy, ls = flow.generate_decoys(test_ds_dist, device=DEVICE)
    ms       = ms[::SUBSAMPLE_STEP]
    ds_decoy = ds_decoy[::SUBSAMPLE_STEP]
    ls       = ls[::SUBSAMPLE_STEP]

    # plot_score_distribution_with_decoys для каждого класса
    for i in range(NUM_CLASSES):
        plot_score_distribution_with_decoys(
            train_scores[:, i],   # train scores для класса i
            train_labels,
            ms[:, i],             # test scores для класса i
            ls,
            ds_decoy[:, i],       # decoy scores для класса i
            filename  = f"BCSS/figures/diagnostics/scores_dist_class{i}",
            title     = f"Score Distribution with Decoys — Class {i}",
            xlim      = (-10, 10),
            show_kde  = True,
            class_id  = i,
        )
    print(f"✓ Score distributions сохранены для {NUM_CLASSES} классов")

except Exception as e:
    print(f"✗ Ошибка при построении distributions: {e}")

Device    : cuda:1
Baselines : ['ATC', 'ATC-NE', 'AC', 'DOC', 'COT']

LOADING TRAINING DATA


/tmp/ipykernel_54138/3190543947.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data     = torch.load(PATH_DATA)


Train scores shape: (3394245, 5)
  Class 0: 1000000 samples
  Class 1: 1000000 samples
  Class 2: 776366 samples
  Class 3: 568007 samples
  Class 4: 49872 samples

FLOW MODEL
✓ ScoreShiftFlow defined — single flow over full score vector
✓ Flow loaded from BCSS/bcss_score_shift_flow_new.pth

Подготовка source данных для baselines...


/tmp/ipykernel_54138/3190543947.py:181: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  flow.load_state_dict(torch.load(FLOWS_PATH, map_location=DEVICE))


KeyboardInterrupt: 